In [1]:
#hard comparisoin, 400 vs 5160 parameters

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os, glob, tarfile
from pathlib import Path 

tar_path = "global/homes/a/aletesi/PaarT/Data/JetClass_Pythia_val_5M.tar"
extract_dir = "global/homes/a/aletesi/PaarT/Data/JetClass_Pythia_val_5M"

files = glob.glob(os.path.join(Path("/global/homes/a/aletesi/PaarT/Data/JetClass_Pythia_val_5M/val_5M"), "*.root"))
from torch.utils.data import Dataset
!pip install awkward uproot vector
from particle_transformer.dataloader import read_file

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/torchvision-0.21.0+7af6987-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/setuptools-75.8.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/pillow-11.1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package install

In [3]:
all_x_parts = []
all_ys = []

for file in files:
    x_part, x_jets, y = read_file(
        file,
        max_num_particles=8,
        particle_features=['part_pt', 'part_eta', 'part_phi', 'part_energy'],
        jet_features=['jet_pt', 'jet_eta', 'jet_phi', 'jet_energy'],
        labels=[
            'label_QCD', 'label_Hbb', 'label_Hcc', 'label_Hgg', 'label_H4q',
            'label_Hqql', 'label_Zqq', 'label_Wqq', 'label_Tbqq', 'label_Tbl',
        ]
    )
    all_x_parts.append(torch.tensor(x_part, dtype=torch.float32)[:20,:,:])
    all_ys.append(torch.tensor(y, dtype=torch.float32)[:20,:])

x_all = torch.cat(all_x_parts, dim=0)
y_all = torch.cat(all_ys, dim=0)
print(x_all.shape, y_all.shape)

class JetDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = JetDataset(x_all, y_all)

from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

torch.Size([1000, 4, 8]) torch.Size([1000, 10])


In [4]:
!pip install tensorcircuit
from QParT import ParT

model = ParT(
    in_dim=4,           # part_pt, eta, phi, energy
    embed_dim=10,
    n_heads=2,
    depth=2,
    class_depth=2,
    num_classes=10
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"{name:40s} | {p.numel():,} params | shape {tuple(p.shape)}")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"Trainable parameters: {total_params:,}")

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/torchvision-0.21.0+7af6987-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/setuptools-75.8.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /global/common/software/nersc9/pytorch/2.6.0/lib/python3.12/site-packages/pillow-11.1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package install

Please first ``pip install -U qiskit`` to enable related functionality in translation module
Please first ``pip install -U cirq`` to enable related functionality in translation module


class_token                              | 10 params | shape (1, 1, 10)
tokenizer.proj.weight                    | 40 params | shape (10, 4)
tokenizer.proj.bias                      | 10 params | shape (10,)
U_encoder.net.0.weight                   | 256 params | shape (64, 4, 1, 1)
U_encoder.net.1.weight                   | 64 params | shape (64,)
U_encoder.net.1.bias                     | 64 params | shape (64,)
U_encoder.net.3.weight                   | 4,096 params | shape (64, 64, 1, 1)
U_encoder.net.4.weight                   | 64 params | shape (64,)
U_encoder.net.4.bias                     | 64 params | shape (64,)
U_encoder.net.6.weight                   | 4,096 params | shape (64, 64, 1, 1)
U_encoder.net.7.weight                   | 64 params | shape (64,)
U_encoder.net.7.bias                     | 64 params | shape (64,)
U_encoder.net.9.weight                   | 128 params | shape (2, 64, 1, 1)
blocks.0.ln1.weight                      | 10 params | shape (10,)
blocks.0.ln1.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for batch_idx, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)  # shape [batch, 10]

        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [ ]:
from torch.nn.functional import sigmoid, softmax

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        labels = torch.argmax(y, dim=1)          # convert one-hot to class id
        preds = torch.argmax(outputs, dim=1)     # predicted class
        correct += (preds == labels).sum().item()
        total += y.size(0)
accuracy = correct / total
print(f"Accuracy on full dataset: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

model.eval()
all_outputs = []
all_targets = []

with torch.no_grad():
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        all_outputs.append(outputs.cpu())
        all_targets.append(y.cpu())

# Concatenate batches
all_outputs = torch.cat(all_outputs, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Apply sigmoid (if using BCEWithLogitsLoss)
probs = sigmoid(all_outputs).numpy()  # shape: (N, C)
true = all_targets.numpy()            # shape: (N, C)

# Compute AUC for each class and average
try:
    auc_macro = roc_auc_score(true, probs, average='macro', multi_class='ovr')
    print(f"Macro-Averaged AUC: {auc_macro:.4f}")
except ValueError as e:
    print("AUC could not be computed:", e)